In [1]:
import numpy as np
import torch
import torch.nn as nn
import random
import optuna
from torch.optim import LBFGS, Adam
from model_components.models import FourierPINNsformer
from model_components.util import *
import scipy.io

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# reproducibility
def set_seed(seed=0):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# analytical solution

mat = scipy.io.loadmat('../convection.mat')
u_true = mat['u'].reshape(101,101)


res, b_left, b_right, b_upper, b_lower = get_data([0, 2*np.pi], [0,1], 51, 51)
res_test, _, _, _, _ = get_data([0, 2*np.pi], [0,1], 101, 101)

res, b_left, b_right, b_upper, b_lower = get_data([0,2*np.pi], [0,1], 51, 51)
res_test, _, _, _, _ = get_data([0,2*np.pi], [0,1], 101, 101)

res = make_time_sequence(res, num_step=5, step=1e-4)
b_left = make_time_sequence(b_left, num_step=5, step=1e-4)
b_right = make_time_sequence(b_right, num_step=5, step=1e-4)
b_upper = make_time_sequence(b_upper, num_step=5, step=1e-4)
b_lower = make_time_sequence(b_lower, num_step=5, step=1e-4)

res = torch.tensor(res, dtype=torch.float32, requires_grad=True).to(device)
b_left = torch.tensor(b_left, dtype=torch.float32, requires_grad=True).to(device)
b_right = torch.tensor(b_right, dtype=torch.float32, requires_grad=True).to(device)
b_upper = torch.tensor(b_upper, dtype=torch.float32, requires_grad=True).to(device)
b_lower = torch.tensor(b_lower, dtype=torch.float32, requires_grad=True).to(device)

x_res, t_res = res[:,:,0:1], res[:,:,1:2]
x_left, t_left = b_left[:,:,0:1], b_left[:,:,1:2]
x_right, t_right = b_right[:,:,0:1], b_right[:,:,1:2]
x_upper, t_upper = b_upper[:,:,0:1], b_upper[:,:,1:2]
x_lower, t_lower = b_lower[:,:,0:1], b_lower[:,:,1:2]

res_test = make_time_sequence(res_test, num_step=5, step=1e-4) 
res_test = torch.tensor(res_test, dtype=torch.float32, requires_grad=True).to(device)
x_test, t_test = res_test[:,:,0:1], res_test[:,:,1:2]


kernel_size = 300

D1 = kernel_size
D2 = len(x_left)
D3 = len(x_lower)


def compute_ntk(J1, J2):
    Ker = torch.matmul(J1, torch.transpose(J2, 0, 1))
    return Ker

smallest_rl1 = 1e10  # Initialize a large value to track the smallest L1 error

# objective for optuna

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

set_seed(0)

# sample hyperparameters
d_hidden = 256
d_model = 128
mapping_size = 32

# build model
model = FourierPINNsformer(
    d_out=1,
    d_hidden=d_hidden,
    d_model=d_model,
    N=1,
    heads=2,
    d_in=2,
    mapping_size=mapping_size,
    x_range=(0.0, 2*torch.pi),
    t_range=(0.0, 1.0),
    activation_function='wave_act'
).to(device)

# init weights
for m in model.modules():
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)
        
        
n_params = get_n_params(model)


model.load_state_dict(torch.load('saves/conv_spformer_trial_5.pth', map_location=device))

    
# evaluate L1 error
model.eval()
with torch.no_grad():
    pred = model(x_test, t_test)[:,0].cpu().numpy().reshape(101,101)
rl1 = np.sum(np.abs(u_true - pred)) / np.sum(np.abs(u_true))
rl2 = np.sqrt(np.sum((u_true - pred) ** 2) / np.sum(u_true ** 2))


# run study

print(f"L1 error: {rl1:.6f}, L2 error: {rl2:.6f}, n_params: {n_params}")

/home/ubuntu/decoder-only-pinnsformer/pinns/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


L1 error: 0.014741, L2 error: 0.017551, n_params: 305551
